### Importation des bibliothèques

Les bibliothèques nécessaires sont importées afin de manipuler les données et réaliser les visualisations.

In [ ]:
%pip install numpy pandas pandas seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

### Chargement des données

Le fichier `mesures_capteurs.csv` est importé dans un DataFrame nommé `df`. Ce DataFrame sera utilisé tout au long de l'atelier.

In [ ]:
df = pd.read_csv("../data/mesures_capteurs.csv")

df

### Exploration des données

Une première inspection du DataFrame permet de vérifier que les données ont été correctement chargées et d'observer leur structure.

In [ ]:
df.head()

df.info()

df.describe()

df.isnull().sum()

### 1.1 : Vérification des doublons

Avant de poursuivre l'analyse des données, il est important de vérifier si le DataFrame contient des lignes dupliquées. Les doublons peuvent fausser les statistiques et les visualisations réalisées par la suite.

In [ ]:
# Nombre de lignes dupliquées
nb_doublons = df.duplicated().sum()

print(f"Nombre de doublons : {nb_doublons}")

### 1.2 : Suppression des doublons

Si des doublons sont présents, ils sont supprimés afin de conserver une seule occurrence de chaque observation. Une nouvelle vérification est ensuite réalisée pour confirmer que le nettoyage a bien été effectué.

In [ ]:
# Suppression des doublons
df = df.drop_duplicates()

# Vérification après suppression
nb_doublons = df.duplicated().sum()

print(f"Nombre de doublons après suppression : {nb_doublons}")

### 2.1 : Définition de la cible et des caractéristiques

Dans cette étape, la colonne `etat` est définie comme la variable cible (`y`), c'est-à-dire la valeur que le modèle devra prédire.

Les colonnes `temperature`, `humidite`, `pression` et `consommation` constituent les variables explicatives (`X`) qui serviront à entraîner le modèle.

In [ ]:
# Variable cible
y = df["etat"]

# Variables explicatives
X = df[["temperature", "humidite", "pression", "consommation"]]

### 2.2 : Aperçu des données

Les cinq premières lignes de `X` et de `y` sont affichées afin de vérifier que les variables ont été correctement sélectionnées.

In [ ]:
print("Variables explicatives (X) :")
display(X.head())

print("Variable cible (y) :")
display(y.head())

### 2.3 : Type du problème

La variable cible `etat` contient des catégories (par exemple : `OK`, `ALERTE` et `ERREUR`). Le modèle devra prédire une classe à partir de plusieurs caractéristiques.

Il s'agit donc d'un problème de **classification**.

In [ ]:
print("Type du problème :", "Classification")

print("\nClasses présentes dans la cible :")
print(y.unique())

### 3.1 : Découpage Train/Test

Les données sont divisées en deux ensembles :

- un ensemble d'entraînement (`train`) utilisé pour apprendre le modèle ;
- un ensemble de test (`test`) utilisé pour évaluer ses performances.

Le découpage est réalisé en réservant 20 % des données au test. La reproductibilité est assurée grâce à `random_state`, tandis que `stratify` permet de conserver la même répartition des classes dans les deux ensembles.

In [ ]:
# Vérifier les stats de NaN

df.isnull().sum()

In [ ]:
# suppresion des valeurs manquantes

df = df.dropna()

# Redéfinir X et y
X = df[["temperature", "humidite", "pression", "consommation"]]
y = df["etat"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### 3.2 : Vérifier du découpage

Les dimensions des ensembles obtenus sont affichées afin de vérifier que le découpage a été correctement réalisé.

In [ ]:
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test  :", X_test.shape)

print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test  :", y_test.shape)

### 4.1 : Vérifier des valeurs manquantes

Avant d'entraîner un modèle de Machine Learning, il est important de vérifier si les variables explicatives contiennent des valeurs manquantes. Ces valeurs devront être traitées afin d'éviter des erreurs lors de l'entraînement du modèle.

In [ ]:
# Nombre de valeurs manquantes par variable
X_train.isnull().sum()

In [ ]:
print("Valeurs manquantes dans X_train :")
print(X_train.isnull().sum())

print("\nValeurs manquantes dans X_test :")
print(X_test.isnull().sum())

### 4.2 : Sélection de l'imputeur

L'imputeur `SimpleImputer` est utilisé pour remplacer automatiquement les valeurs manquantes. La stratégie choisie est la médiane.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

### 4.3 : Choix de la médiane

La médiane est moins sensible aux valeurs extrêmes que la moyenne. Elle permet donc de remplacer les valeurs manquantes sans être fortement influencée par d'éventuelles observations atypiques présentes dans les données.

### 4.4 : Apprentissage de l'imputeur

L'imputeur est ajusté uniquement sur les données d'entraînement (`X_train`). Il calcule la médiane de chaque variable afin de remplacer les valeurs manquantes.

In [ ]:
imputer.fit(X_train)

# Médianes calculées
print(imputer.statistics_)

### 4.5 : Remplacement des valeurs manquantes

Les ensembles d'entraînement et de test sont transformés à l'aide de l'imputeur. Les valeurs manquantes sont remplacées par les médianes calculées sur `X_train`.

In [ ]:
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print(X_train_imputed, X_test_imputed)

### 5.1 : Mise à l'échelle avec StandardScaler

Les variables de notre jeu de données n'ont pas nécessairement les mêmes unités ni les mêmes ordres de grandeur.

Nous utilisons `StandardScaler` de Scikit-learn pour standardiser les données après l'imputation des valeurs manquantes.

La standardisation transforme chaque variable selon la formule :

$$z = \frac{x - \mu}{\sigma}$$

où :
- $\mu$ représente la moyenne de la variable ;
- $\sigma$ représente son écart-type.

Le `StandardScaler` sera ajusté uniquement sur `X_train_imputed`, puis utilisé pour transformer les données d'entraînement et de test.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

### 5.2 : Pourquoi standardiser les données ?

La standardisation est importante pour le modèle KNN car celui-ci utilise les distances entre les observations pour déterminer les voisins les plus proches.

Si une variable possède des valeurs beaucoup plus grandes qu'une autre, elle peut avoir une influence disproportionnée sur le calcul des distances.

La standardisation permet donc de placer les différentes variables sur une échelle comparable, avec une moyenne proche de 0 et un écart-type proche de 1.

Cela permet au modèle KNN de prendre en compte les caractéristiques de manière plus équilibrée.

In [ ]:
print("StandardScaler sélectionné pour la mise à l'échelle des données.")

### 5.3 : Paramètres du StandardScaler

Le `StandardScaler` doit être ajusté uniquement sur les données d'entraînement.

La méthode `fit()` permet de calculer, pour chaque variable, les paramètres nécessaires à la standardisation :

- la moyenne ;
- l'écart-type.

Ces paramètres seront ensuite utilisés pour transformer les données d'entraînement et de test.

In [ ]:
scaler.fit(X_train_imputed)

print("Moyennes :")
print(scaler.mean_)

print("\nÉcarts-types :")
print(scaler.scale_)

### 5.4 : Transformation des données

Le `StandardScaler` étant maintenant ajusté sur `X_train_imputed`, nous pouvons transformer les données d'entraînement et de test.

La méthode `transform()` utilise les paramètres calculés à partir de `X_train_imputed`.

Nous obtenons ainsi :

- `X_train_scaled` : les données d'entraînement standardisées ;
- `X_test_scaled` : les données de test standardisées.

Les paramètres ne sont donc pas recalculés sur les données de test.

In [ ]:
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("X_train_scaled :")
print(X_train_scaled[:5])

print("\nX_test_scaled :")
print(X_test_scaled[:5])

### 6.1 : Sélection du modèle KNN

Le modèle **K-Nearest Neighbors (KNN)** est choisi pour réaliser la classification des états des capteurs.

Le principe de KNN consiste à rechercher les **k voisins les plus proches** d'une nouvelle observation, puis à lui attribuer la classe la plus représentée parmi ces voisins.

Dans cet atelier, nous choisissons **k = 5**, ce qui signifie que les cinq voisins les plus proches seront pris en compte pour effectuer la prédiction.

In [25]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)